# PFC -> FNO: generate, train, save (all-in-one)

**One notebook. Set the runtime to GPU, then `Runtime -> Run all`.**
It will:
1. generate the grain-rich PFC dataset (90 runs, `n_seeds=30`),
2. save the dataset to your Google Drive,
3. train the FNO on it,
4. save the trained model + evaluation figures to Drive,
5. show the rollout figures and a results table.

Set the runtime first: **Runtime -> Change runtime type -> T4 GPU**.
(Generation runs on CPU and is quick; only training needs the GPU.)
A Google Drive popup appears at the first cell -- approve it.


### 1. Mount Drive + clone repo + install everything

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard

### 2. Confirm a GPU is attached (training is 10-30x slower without one)

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU OK:", torch.cuda.get_device_name(0))

## Part A — Generate the grain-rich dataset
90 runs (5 r-values x 3 n0 x 3 seed types x 2 noise realizations), `n_seeds=30`
so multi-seed runs build a real grain-boundary network. ~1 minute on CPU.
Every run is sanity-checked (mass / energy / NaN).

In [ ]:
!python generate_dataset.py --sweep --config pfc_config_grains.yaml --parallel --max-workers 2

### 3. Verify the dataset (every run should be `ok`)

In [ ]:
import pandas as pd
m = pd.read_csv("data_pfc_grains/manifest.csv")
print(m["status"].value_counts())
print(f"{len(m)} runs | seed types: {sorted(m['seed_type'].unique())} | "
      f"ensembles: {sorted(m['ensemble'].unique())}")
m.head()

### 4. Save the dataset to Drive (so you never have to regenerate it)

In [ ]:
!zip -r -q pfc_dataset_grains.zip data_pfc_grains
!cp pfc_dataset_grains.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/pfc_dataset_grains.zip

## Part B — Train the FNO
Points the training config at the new dataset. `split_by: config` keeps both
noise realizations of one (r, n0, seed_type) in the same split (no leakage).
The other settings (delta-weighted loss, 4-step rollout training, modes=24)
are already in `config.yaml`.

In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["data_dir"] = "data_pfc_grains"
cfg["data"]["split_by"] = "config"
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
print("data:", cfg["data"])
print("model:", cfg["model"])
print("loss/rollout:", {k: cfg["train"][k] for k in
      ["loss_weight_mode","loss_alpha","rollout_steps"]})

### 5. Train (early-stops automatically; live per-batch progress shown)

In [ ]:
!python train_fno.py --config config.yaml

### 6. Save the trained model to Drive immediately

In [ ]:
!cp runs/fno_baseline/best.pt /content/drive/MyDrive/fno_grains_best.pt
print("Checkpoint saved: MyDrive/fno_grains_best.pt")

## Part C — Evaluate
One-step MSE, rollout MSE, mass error, the 8 worst trajectories (each figure
labeled with its r / n0 / seed_type), and MSE grouped by seed_type and r.

In [ ]:
!python evaluate_fno.py --config config.yaml --checkpoint runs/fno_baseline/best.pt

from IPython.display import Image, display
import glob
for png in sorted(glob.glob("runs/fno_baseline/eval/*.png")):
    display(Image(png))

### 7. Results table: which regime is still hardest?

In [ ]:
import pandas as pd
df = pd.read_csv("runs/fno_baseline/eval/per_trajectory.csv")
print("Per-trajectory (worst first):")
display(df.sort_values("rollout_mse", ascending=False))
print("\nMean rollout MSE by seed_type:")
display(df.groupby("seed_type")["rollout_mse"].agg(["mean","count"]))
print("\nMean rollout MSE by r:")
display(df.groupby("r")["rollout_mse"].agg(["mean","count"]))

### 8. Save the evaluation figures to Drive too

In [ ]:
!zip -r -q fno_grains_eval.zip runs/fno_baseline/eval
!cp fno_grains_eval.zip /content/drive/MyDrive/
print("Saved: MyDrive/fno_grains_eval.zip  +  fno_grains_best.pt  +  pfc_dataset_grains.zip")